# Feature Engineering: The Art of Crafting Better Features

> **Key Insight:** Feature engineering is arguably the most important step in the machine learning pipeline. Even a powerful algorithm will perform poorly with bad features, whereas a weak algorithm can perform well with strong features.

Feature engineering is the process of using domain knowledge to create, transform, and select features that make machine learning algorithms work more effectively. This notebook covers the **four main areas** of feature engineering.

---

## The Four Pillars of Feature Engineering

| Category | Description | Key Techniques |
|----------|-------------|----------------|
| **Feature Transformation** | Modifying existing features | Scaling, Encoding, Imputation, Outlier Handling |
| **Feature Construction** | Creating new features | Domain-based combinations, Aggregations |
| **Feature Selection** | Choosing relevant features | Filter, Wrapper, Embedded methods |
| **Feature Extraction** | Generating new features programmatically | PCA, LDA, Autoencoders |

> 💡 **Remember:** The quality of your features directly determines the ceiling of your model's performance!

---

# Part 1: Feature Transformation

> **Feature Transformation** involves modifying existing features to improve model performance. This includes handling missing values, encoding categorical variables, detecting outliers, and scaling features.

## Overview of Transformation Techniques

| Technique | Purpose | When to Use |
|-----------|---------|-------------|
| Missing Value Imputation | Handle incomplete data | When data has NaN/null values |
| Categorical Encoding | Convert text to numbers | When features are non-numerical |
| Outlier Detection | Handle extreme values | When data has unusual points |
| Feature Scaling | Standardize value ranges | When features have different scales |

## 1.1 Missing Values Imputation

> Missing data is a common problem that can significantly impact model performance. **Imputation** is the process of replacing missing values with substitute values.

### Two Main Strategies:

| Strategy | Description | Pros | Cons |
|----------|-------------|------|------|
| **Deletion** | Remove rows/columns with missing data | Simple, no assumptions | Loss of data |
| **Imputation** | Fill in missing values | Preserves data | May introduce bias |

---

### Deletion Methods:

```python
# Remove rows with any missing values
df_clean = df.dropna()

# Remove rows where specific column is missing
df_clean = df.dropna(subset=['Age'])

# Remove columns with missing values
df_clean = df.dropna(axis=1)

# Remove columns with more than 50% missing
threshold = len(df) * 0.5
df_clean = df.dropna(thresh=threshold, axis=1)
```

> ⚠️ **Warning:** Only use deletion when missing data is minimal (< 5%), otherwise you lose valuable information!

---

### Imputation Methods:

| Method | Use Case | Code |
|--------|----------|------|
| **Mean** | Numerical, normally distributed | `df['col'].fillna(df['col'].mean())` |
| **Median** | Numerical, skewed data | `df['col'].fillna(df['col'].median())` |
| **Mode** | Categorical data | `df['col'].fillna(df['col'].mode()[0])` |
| **Constant** | When domain-specific value exists | `df['col'].fillna(0)` |

### Example Implementation:

```python
import pandas as pd
from sklearn.impute import SimpleImputer

# Using Pandas
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Using Scikit-learn (better for pipelines)
imputer_mean = SimpleImputer(strategy='mean')
imputer_mode = SimpleImputer(strategy='most_frequent')

df[['Age']] = imputer_mean.fit_transform(df[['Age']])
df[['Embarked']] = imputer_mode.fit_transform(df[['Embarked']])
```

### Advanced Imputation Techniques:

| Technique | Description | When to Use |
|-----------|-------------|-------------|
| **KNN Imputation** | Uses similar samples to fill missing | When data has local patterns |
| **Iterative Imputation** | Models each feature as a function of others | When features are correlated |
| **Group-based** | Impute based on group statistics | When groups have different distributions |

```python
from sklearn.impute import KNNImputer

# KNN Imputer - uses 5 nearest neighbors by default
knn_imputer = KNNImputer(n_neighbors=5)
df_imputed = pd.DataFrame(
    knn_imputer.fit_transform(df),
    columns=df.columns
)
```

> 💡 **Pro Tip:** Always investigate **why** data is missing before choosing an imputation method. Missing data can be:
> - **MCAR** (Missing Completely at Random): Safe to impute
> - **MAR** (Missing at Random): Depends on other variables
> - **MNAR** (Missing Not at Random): Most problematic, value itself affects missingness

## 1.2 Handling Categorical Values

> Most machine learning algorithms require **numerical input**. Categorical encoding converts non-numerical data into formats that algorithms can process.

### Types of Categorical Data:

| Type | Description | Examples | Encoding Method |
|------|-------------|----------|----------------|
| **Nominal** | No inherent order | Color (Red, Blue), Gender | One-Hot Encoding |
| **Ordinal** | Has meaningful order | Size (S, M, L), Education Level | Label Encoding |
| **Binary** | Only two categories | Yes/No, Male/Female | Binary (0/1) |

---

### Encoding Techniques:

#### 1. Label Encoding (for Ordinal Data)

```python
from sklearn.preprocessing import LabelEncoder

# Convert categories to numbers: S=0, M=1, L=2
le = LabelEncoder()
df['Size_encoded'] = le.fit_transform(df['Size'])

# Manual ordinal encoding (when order matters)
size_mapping = {'S': 0, 'M': 1, 'L': 2, 'XL': 3}
df['Size_encoded'] = df['Size'].map(size_mapping)
```

#### 2. One-Hot Encoding (for Nominal Data)

```python
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# Using Pandas (simplest approach)
df_encoded = pd.get_dummies(df, columns=['Color'])
# Creates: Color_Red, Color_Blue, Color_Green

# Using Scikit-learn (better for pipelines)
ohe = OneHotEncoder(sparse=False, drop='first')  # drop='first' avoids dummy trap
encoded = ohe.fit_transform(df[['Color']])
```

### One-Hot Encoding Visualization:

```
Original:           After One-Hot Encoding:
┌─────────┐         ┌─────────┬─────────┬─────────┐
│  Color  │         │Color_Red│Color_Blue│Color_Green│
├─────────┤         ├─────────┼─────────┼─────────┤
│   Red   │   →     │    1    │    0    │    0    │
│  Blue   │   →     │    0    │    1    │    0    │
│  Green  │   →     │    0    │    0    │    1    │
│   Red   │   →     │    1    │    0    │    0    │
└─────────┘         └─────────┴─────────┴─────────┘
```

---

### Advanced Encoding Techniques:

| Technique | Description | Use Case |
|-----------|-------------|----------|
| **Target Encoding** | Replace category with target mean | High cardinality features |
| **Frequency Encoding** | Replace with category frequency | When frequency matters |
| **Binary Encoding** | Combine label + one-hot | Many categories |

```python
# Target Encoding Example
target_means = df.groupby('City')['Survived'].mean()
df['City_encoded'] = df['City'].map(target_means)

# Frequency Encoding Example
freq = df['City'].value_counts(normalize=True)
df['City_freq'] = df['City'].map(freq)
```

> ⚠️ **The Dummy Variable Trap:** When using one-hot encoding, drop one column to avoid multicollinearity (use `drop='first'` in sklearn or `drop_first=True` in pandas).

> 💡 **Pro Tip:** For high cardinality features (many unique values), consider target encoding or embedding layers instead of one-hot encoding to avoid dimensionality explosion!

## 1.3 Outlier Detection

> **Outliers** are data points that differ significantly from other observations. They can negatively impact model performance by skewing statistics and violating model assumptions.

### Impact of Outliers:

| Effect | Description |
|--------|-------------|
| **Mean shift** | Extreme values pull the mean towards them |
| **Increased variance** | Spread of data appears larger than reality |
| **Model bias** | Linear models try to fit outliers, hurting overall performance |
| **Distance distortion** | KNN, K-means affected by extreme distances |

---

### Detection Methods:

#### 1. Statistical Methods (Z-Score)

```python
from scipy import stats
import numpy as np

# Z-score: values beyond ±3 standard deviations are outliers
z_scores = np.abs(stats.zscore(df['column']))
outliers = df[z_scores > 3]

# Remove outliers
df_clean = df[z_scores <= 3]
```

| Z-Score Threshold | Percentage of Data Kept |
|-------------------|------------------------|
| ±2 | ~95.4% |
| ±3 | ~99.7% |

#### 2. IQR Method (Interquartile Range)

```python
# Calculate IQR
Q1 = df['column'].quantile(0.25)
Q3 = df['column'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter outliers
df_clean = df[(df['column'] >= lower_bound) & (df['column'] <= upper_bound)]
```

### Visual Representation of IQR:

```
         ◄─────────── IQR ───────────►
    ┌────────┬──────────────────┬────────┐
    │        │                  │        │
────┼────────┼──────────────────┼────────┼────
    │        │                  │        │
    └────────┴──────────────────┴────────┘
    ↑        ↑         ↑        ↑        ↑
 Lower     Q1      Median     Q3     Upper
 Bound                               Bound
(Q1-1.5*IQR)                    (Q3+1.5*IQR)

◄──── Outliers ────►            ◄──── Outliers ────►
```

---

### Handling Strategies:

| Strategy | Description | When to Use |
|----------|-------------|-------------|
| **Remove** | Delete outlier rows | When outliers are errors |
| **Cap/Winsorize** | Clip to threshold values | Preserve data, reduce impact |
| **Transform** | Apply log/sqrt transformation | When distribution is skewed |
| **Separate Model** | Build different model for outliers | When outliers are valid but different |
| **Keep** | Do nothing | When outliers carry important information |

```python
# Capping/Winsorizing
df['column'] = df['column'].clip(lower=lower_bound, upper=upper_bound)

# Log transformation (reduces impact of large values)
df['log_column'] = np.log1p(df['column'])  # log1p handles zeros
```

> 💡 **Pro Tip:** Always investigate outliers before removing them! They might be:
> - **Data entry errors** → Remove or correct
> - **Measurement errors** → Remove
> - **Genuine extreme values** → Consider keeping or separate modeling

## 1.4 Feature Scaling

> **Feature Scaling** standardizes the range of independent variables. This is crucial when features have different units or scales, ensuring no single feature dominates the model.

### Why Scaling Matters:

| Without Scaling | With Scaling |
|-----------------|-------------|
| Age: 0-100 | Age: 0-1 |
| Salary: 20,000-500,000 | Salary: 0-1 |
| ❌ Salary dominates | ✓ Equal contribution |

### Algorithms Affected by Scaling:

| Needs Scaling | Scale-Invariant |
|---------------|----------------|
| KNN, K-Means | Decision Trees |
| SVM | Random Forest |
| Neural Networks | Gradient Boosting |
| Linear/Logistic Regression | Naive Bayes |
| PCA | |

---

### Scaling Techniques:

#### 1. Min-Max Scaling (Normalization)

Scales values to a fixed range, typically [0, 1].

$$X_{scaled} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[['Age', 'Salary']] = scaler.fit_transform(df[['Age', 'Salary']])

# Manual implementation
df['Age_scaled'] = (df['Age'] - df['Age'].min()) / (df['Age'].max() - df['Age'].min())
```

| Pros | Cons |
|------|------|
| Bounded range | Sensitive to outliers |
| Preserves zero entries | New data may fall outside [0,1] |
| Good for neural networks | |

---

#### 2. Standardization (Z-Score Normalization)

Transforms data to have mean=0 and standard deviation=1.

$$X_{scaled} = \frac{X - \mu}{\sigma}$$

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[['Age', 'Salary']] = scaler.fit_transform(df[['Age', 'Salary']])

# Manual implementation
df['Age_scaled'] = (df['Age'] - df['Age'].mean()) / df['Age'].std()
```

| Pros | Cons |
|------|------|
| Less affected by outliers | No bounded range |
| Works with most algorithms | Doesn't preserve zero entries |
| Assumes normal distribution | |

---

### Comparison of Scaling Methods:

| Method | Range | Best For |
|--------|-------|----------|
| **Min-Max** | [0, 1] | Neural networks, image pixels |
| **Standardization** | No fixed range | Most ML algorithms |
| **Robust Scaler** | No fixed range | Data with outliers |
| **MaxAbs Scaler** | [-1, 1] | Sparse data |

```python
from sklearn.preprocessing import RobustScaler

# Robust Scaler - uses median and IQR, less sensitive to outliers
robust_scaler = RobustScaler()
df[['Age']] = robust_scaler.fit_transform(df[['Age']])
```

> ⚠️ **Important:** Always fit the scaler on **training data only**, then transform both train and test sets:

```python
# Correct approach
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Wrong: fitting on all data causes data leakage!
# scaler.fit(X)  ❌
```

> 💡 **Pro Tip:** Use `StandardScaler` as your default choice. Switch to `MinMaxScaler` for neural networks or `RobustScaler` when dealing with outliers.

---

# Part 2: Feature Construction

> **Feature Construction** (also called Feature Creation) involves creating new features from existing ones based on **domain knowledge** or intuition. These engineered features often capture relationships that the original features cannot express.

### Why Construct New Features?

| Benefit | Example |
|---------|--------|
| Capture domain knowledge | BMI = Weight / Height² |
| Reveal hidden patterns | Age Group from Age |
| Combine correlated features | Total = Feature1 + Feature2 |
| Create interaction effects | Price per Square Foot |

---

### Common Construction Techniques:

#### 1. Arithmetic Combinations

```python
# Titanic Dataset Example
# Create 'Family Size' from 'SibSp' (siblings/spouses) and 'Parch' (parents/children)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # +1 for the person

# Is the person traveling alone?
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
```

#### 2. Binning/Discretization

```python
# Convert continuous to categorical
df['AgeGroup'] = pd.cut(df['Age'], 
                        bins=[0, 12, 18, 35, 60, 100],
                        labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior'])

# Equal-frequency binning
df['Income_Quartile'] = pd.qcut(df['Income'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])
```

#### 3. Date/Time Features

```python
# Extract components from datetime
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek  # 0=Monday
df['IsWeekend'] = (df['DayOfWeek'] >= 5).astype(int)
df['Quarter'] = df['Date'].dt.quarter

# Time since an event
df['DaysSinceSignup'] = (pd.Timestamp.today() - df['SignupDate']).dt.days
```

---

### Advanced Construction Examples:

| Domain | Original Features | Constructed Feature |
|--------|-------------------|--------------------|
| **E-commerce** | Price, Quantity | Total_Spent = Price × Quantity |
| **Real Estate** | Price, Area | Price_per_SqFt = Price / Area |
| **Health** | Weight (kg), Height (m) | BMI = Weight / Height² |
| **Finance** | Debt, Income | Debt_to_Income = Debt / Income |
| **Titanic** | SibSp, Parch | FamilySize = SibSp + Parch + 1 |

```python
# Text-based feature construction
df['Name_Length'] = df['Name'].apply(len)
df['Has_Title'] = df['Name'].str.contains('Mr\\.|Mrs\\.|Dr\\.').astype(int)
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Polynomial features
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df[['Age', 'Fare']])
# Creates: Age, Fare, Age², Fare², Age×Fare
```

> 💡 **Pro Tip:** The best features come from **domain expertise**. Always consult with subject matter experts or research the problem domain to discover meaningful feature combinations!

> ⚠️ **Warning:** Don't over-engineer features! Too many features can lead to overfitting and increased training time. Always validate that new features improve model performance.

---

# Part 3: Feature Selection

> **Feature Selection** is the process of choosing a subset of relevant features from a larger set. This improves model performance, reduces training time, and makes models more interpretable.

### Why Select Features?

| Problem | Solution via Selection |
|---------|------------------------|
| **Curse of Dimensionality** | Fewer features = better generalization |
| **Overfitting** | Remove noise features |
| **Slow Training** | Less computation required |
| **Poor Interpretability** | Focus on most important features |

---

### Real-World Example: MNIST Dataset

The MNIST handwritten digit dataset has 28×28 = 784 pixels (features) per image.

```
Original Image (28×28 = 784 features):
┌────────────────────────────┐
│ □ □ □ □ □ □ □ □ □ □ □ □ □ │
│ □ □ □ □ □ ■ ■ ■ □ □ □ □ □ │  ← Border pixels are usually
│ □ □ □ □ ■ ■ ■ ■ ■ □ □ □ □ │     empty (always 0)
│ □ □ □ ■ ■ □ □ ■ ■ □ □ □ □ │
│ □ □ □ ■ ■ □ □ ■ ■ □ □ □ □ │  ← Center pixels carry most
│ □ □ □ □ □ □ □ ■ ■ □ □ □ □ │     of the information
│ □ □ □ □ □ □ ■ ■ □ □ □ □ □ │
│ □ □ □ □ □ ■ ■ □ □ □ □ □ □ │
│ □ □ □ □ ■ ■ ■ ■ ■ ■ □ □ □ │
│ □ □ □ □ □ □ □ □ □ □ □ □ □ │
└────────────────────────────┘

Many border pixels are ALWAYS 0 → They can be removed!
Selecting only informative pixels: 784 → ~200 features
```

---

### Three Categories of Feature Selection:

| Category | Method | Pros | Cons |
|----------|--------|------|------|
| **Filter** | Statistical tests independent of model | Fast | May miss feature interactions |
| **Wrapper** | Uses model performance to evaluate | Considers interactions | Computationally expensive |
| **Embedded** | Selection built into model | Efficient | Model-specific |

---

### 1. Filter Methods

```python
from sklearn.feature_selection import SelectKBest, chi2, f_classif

# Select top k features based on statistical tests
selector = SelectKBest(score_func=f_classif, k=10)
X_selected = selector.fit_transform(X, y)

# Get feature scores
scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': selector.scores_
}).sort_values('Score', ascending=False)

# Correlation-based selection
# Remove features with low correlation to target
correlations = df.corr()['target'].abs().sort_values(ascending=False)
selected_features = correlations[correlations > 0.1].index.tolist()
```

### 2. Wrapper Methods

```python
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

# Recursive Feature Elimination
model = RandomForestClassifier()
rfe = RFE(model, n_features_to_select=10)
X_selected = rfe.fit_transform(X, y)

# Get selected feature names
selected_features = X.columns[rfe.support_].tolist()
```

### 3. Embedded Methods

```python
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestClassifier

# L1 Regularization (Lasso) - automatically sets unimportant features to 0
lasso = LassoCV(cv=5)
lasso.fit(X, y)
important_features = X.columns[lasso.coef_ != 0].tolist()

# Tree-based feature importance
rf = RandomForestClassifier()
rf.fit(X, y)

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)
```

---

### Removing Low-Variance Features:

```python
from sklearn.feature_selection import VarianceThreshold

# Remove features with variance below threshold
selector = VarianceThreshold(threshold=0.01)
X_selected = selector.fit_transform(X)

# For MNIST: removes pixels that are almost always 0
```

> 💡 **Pro Tip:** Start with simple filter methods, then try wrapper/embedded methods if needed. Always validate that feature selection improves your specific model's performance!

---

# Part 4: Feature Extraction

> **Feature Extraction** generates completely new features from existing ones using mathematical transformations, typically reducing dimensionality while preserving important information.

### Key Difference from Selection:

| Feature Selection | Feature Extraction |
|-------------------|--------------------|
| Chooses subset of existing features | Creates new features from combinations |
| Original features remain | Original features are transformed |
| Interpretable | May lose interpretability |
| Example: Keep columns A, B, C | Example: Create PC1, PC2 from A, B, C, D |

---

## 4.1 Principal Component Analysis (PCA)

> PCA is an **unsupervised** technique that transforms features into a new set of uncorrelated variables called **principal components**, ordered by the amount of variance they explain.

### How PCA Works:

```
Original 3D Data:                 After PCA (2D):
                                  
    z ↑                              PC2 ↑
      │    *  *                         │   *  * 
      │  * * *                          │ *  *  * 
      │ *  *                            │*  *  
      │________→ y                      │________→ PC1
     /                                  
    x                             (Maximum variance on PC1)

784 features (MNIST) → 50-100 principal components
(While retaining ~95% of information)
```

### Implementation:

```python
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Step 1: Scale the data (crucial for PCA!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 2: Apply PCA
# Option A: Specify number of components
pca = PCA(n_components=50)
X_pca = pca.fit_transform(X_scaled)

# Option B: Specify variance to retain
pca = PCA(n_components=0.95)  # Keep 95% of variance
X_pca = pca.fit_transform(X_scaled)

# Check explained variance
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.2%}")
```

### Choosing Number of Components:

```python
import matplotlib.pyplot as plt
import numpy as np

# Elbow plot
pca = PCA()
pca.fit(X_scaled)

cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(10, 6))
plt.plot(cumulative_variance)
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% Variance')
plt.legend()
plt.title('PCA - Explained Variance vs Components')
plt.show()
```

| Components | Use Case |
|------------|----------|
| 2-3 | Visualization |
| 95% variance | General dimensionality reduction |
| 99% variance | When accuracy is critical |

---

## 4.2 Linear Discriminant Analysis (LDA)

> LDA is a **supervised** technique that finds linear combinations of features that best separate different classes. Unlike PCA, it uses class labels.

### PCA vs LDA:

| Aspect | PCA | LDA |
|--------|-----|-----|
| **Type** | Unsupervised | Supervised |
| **Goal** | Maximize variance | Maximize class separation |
| **Uses labels?** | No | Yes |
| **Max components** | min(n_features, n_samples) | n_classes - 1 |
| **Best for** | General reduction | Classification problems |

### Visual Comparison:

```
Original 2D Data:              PCA Projection:           LDA Projection:
                               (Max Variance)           (Max Separation)
  Class A: ○○○                                         
  Class B: ●●●                  ○○○●●●                   ○○○    ●●●
     ○ ○                           ↓                        ↓
    ○   ●                     Mixed together!          Well separated!
     ●   ●                                              
       ●
```

### Implementation:

```python
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

# LDA for dimensionality reduction
lda = LinearDiscriminantAnalysis(n_components=2)  # For 3-class problem
X_lda = lda.fit_transform(X, y)  # Note: requires y (labels)

# LDA as classifier
lda_clf = LinearDiscriminantAnalysis()
lda_clf.fit(X_train, y_train)
y_pred = lda_clf.predict(X_test)
```

---

## 4.3 Other Extraction Techniques:

| Technique | Description | Use Case |
|-----------|-------------|----------|
| **t-SNE** | Non-linear visualization | High-dim data visualization |
| **UMAP** | Fast non-linear reduction | Large datasets |
| **Autoencoders** | Neural network compression | Complex non-linear relationships |
| **Factor Analysis** | Assumes latent variables | When features share common causes |

```python
from sklearn.manifold import TSNE

# t-SNE for visualization (typically 2D)
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(X)

# Note: t-SNE is for visualization only, not for modeling!
```

> 💡 **Pro Tip:** Use PCA when you want to reduce dimensions in an unsupervised manner. Use LDA when:
> - You have labeled data
> - Your goal is classification
> - You want maximum class separability

> ⚠️ **Important:** Always scale your data before applying PCA or LDA! Unscaled features with large values will dominate the principal components.

---

## Summary: Feature Engineering at a Glance

| Category | Purpose | Key Techniques |
|----------|---------|----------------|
| **Transformation** | Modify existing features | Imputation, Encoding, Scaling, Outlier handling |
| **Construction** | Create new features | Arithmetic, Binning, Date extraction |
| **Selection** | Choose relevant features | Filter, Wrapper, Embedded methods |
| **Extraction** | Generate new feature space | PCA, LDA, t-SNE |

---

## 📋 Feature Engineering Checklist

```
□ Handle Missing Values
  ├── Understand why data is missing (MCAR/MAR/MNAR)
  ├── Choose imputation method (mean/median/mode/KNN)
  └── Or consider deletion if appropriate

□ Encode Categorical Variables
  ├── Identify nominal vs ordinal
  ├── Apply appropriate encoding (one-hot/label/target)
  └── Watch out for high cardinality

□ Handle Outliers
  ├── Detect using Z-score or IQR
  ├── Investigate before removing
  └── Consider capping or transformation

□ Scale Features
  ├── Fit on training data only
  ├── Choose scaler (Standard/MinMax/Robust)
  └── Apply to both train and test

□ Construct New Features
  ├── Apply domain knowledge
  ├── Create meaningful combinations
  └── Validate improvement

□ Select/Extract Features
  ├── Remove low-variance features
  ├── Use correlation analysis
  └── Apply PCA/LDA if needed
```

---

## Key Takeaways

1. **Features > Algorithms:** Good features can make a weak algorithm perform well; bad features will make even the best algorithm fail

2. **Domain Knowledge is Gold:** The best features often come from understanding the problem domain, not just from automated techniques

3. **Prevent Data Leakage:** Always fit transformers (scalers, encoders, imputers) on training data only

4. **Validate Everything:** Every feature engineering step should be validated against model performance

5. **Less Can Be More:** Fewer, high-quality features often beat many low-quality ones

> 💡 **Remember:** Feature engineering is both art and science. It requires creativity, domain expertise, and systematic experimentation. The best data scientists spend significant time on feature engineering before model training!

---

### Quick Reference: When to Use What

| Scenario | Recommended Approach |
|----------|---------------------|
| Missing categorical data | Mode imputation |
| Missing numerical (normal) | Mean imputation |
| Missing numerical (skewed) | Median imputation |
| Nominal categories | One-Hot encoding |
| Ordinal categories | Label encoding |
| High cardinality | Target encoding |
| Outliers present | Robust scaler |
| Neural networks | MinMax scaling |
| General ML | Standard scaling |
| Visualization | t-SNE / PCA (2-3 components) |
| Classification + reduction | LDA |
| General reduction | PCA (95% variance) |